In [1]:

import os
import glob
import zipfile
import requests

url = "https://www.kaggle.com/api/v1/datasets/download/ronikdedhia/next-word-prediction"

zip_path = "/content/next_word_dataset.zip"
extract_path = "/content/next_word_dataset"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(zip_path, "wb") as f:
    f.write(response.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Dataset downloaded and extracted successfully!")

files = glob.glob(extract_path + "/**/*", recursive=True)
data_files = [
    f for f in files
    if os.path.isfile(f)
    and f.lower().endswith((".txt", ".csv", ".tsv", ".json"))
]

print("\nAvailable dataset files:")
for f in data_files:
    print(f)


Dataset downloaded and extracted successfully!

Available dataset files:
/content/next_word_dataset/1661-0.txt


In [2]:

import pandas as pd
import os

sentences = []

# Select the first available supported text dataset file
if not data_files:
    raise FileNotFoundError(
        "No TXT, CSV, TSV or JSON file found in the dataset."
    )

file_path = data_files[0]
print("Reading:", file_path)

if file_path.lower().endswith(".txt"):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        sentences = f.readlines()

elif file_path.lower().endswith((".csv", ".tsv")):
    separator = "\t" if file_path.lower().endswith(".tsv") else ","
    df = pd.read_csv(file_path, sep=separator, on_bad_lines="skip")

    # Find a text column automatically
    text_columns = df.select_dtypes(include="object").columns.tolist()
    if not text_columns:
        raise ValueError("No text column found in this file.")

    print("Text columns:", list(text_columns))
    sentences = df[text_columns[0]].dropna().astype(str).tolist()

elif file_path.lower().endswith(".json"):
    df = pd.read_json(file_path)
    text_columns = df.select_dtypes(include="object").columns.tolist()

    if not text_columns:
        raise ValueError("No text column found in this JSON file.")

    print("Text columns:", list(text_columns))
    sentences = df[text_columns[0]].dropna().astype(str).tolist()

# Clean empty records and limit the subset
sentences = [s.strip() for s in sentences if s.strip()]
sentences = sentences[:10000]

print("Number of text records:", len(sentences))
print("\nSample records:")
for s in sentences[:5]:
    print(s)


Reading: /content/next_word_dataset/1661-0.txt
Number of text records: 9634

Sample records:
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle
This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included


In [3]:

import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

# Basic text cleaning
clean_sentences = []

for sentence in sentences:
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z0-9\s']", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()

    if sentence:
        clean_sentences.append(sentence)

# Split at sentence/record level BEFORE creating sequences
split_index = int(len(clean_sentences) * 0.90)

train_sentences = clean_sentences[:split_index]
test_sentences = clean_sentences[split_index:]

# Learn the vocabulary from training sentences only
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(train_sentences)

vocab_size = len(tokenizer.word_index) + 1

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))
print("Vocabulary size:", vocab_size)

print("\nWord IDs:")
for word, word_id in list(tokenizer.word_index.items())[:15]:
    print(word, ":", word_id)


Training sentences: 8669
Testing sentences: 964
Vocabulary size: 7653

Word IDs:
<OOV> : 1
the : 2
and : 3
i : 4
to : 5
of : 6
a : 7
in : 8
that : 9
it : 10
he : 11
you : 12
was : 13
his : 14
is : 15


In [4]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

# Keep the vocabulary manageable for Colab
MAX_VOCAB = 12000
tokenizer.num_words = MAX_VOCAB
vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB)

def generate_sequences(texts):
    sequences = []

    for sentence in texts:
        token_ids = tokenizer.texts_to_sequences([sentence])[0]

        # Each longer prefix predicts its next word
        for i in range(1, len(token_ids)):
            sequences.append(token_ids[:i + 1])

    return sequences

train_sequences = generate_sequences(train_sentences)
test_sequences = generate_sequences(test_sentences)

print("Training sequences:", len(train_sequences))
print("Testing sequences:", len(test_sequences))
print("\nExample sequence:", train_sequences[0])


Training sequences: 89620
Testing sequences: 9995

Example sequence: [1961, 4236]


In [5]:

MAX_LEN = 20

def prepare_xy(sequences):
    # Keep only sequences with at least one input and one target
    sequences = [s for s in sequences if len(s) >= 2]

    padded = pad_sequences(
        sequences,
        maxlen=MAX_LEN + 1,
        padding="pre",
        truncating="pre"
    )

    X = padded[:, :-1]
    y = padded[:, -1]

    return X, y

X_train, y_train = prepare_xy(train_sequences)
X_test, y_test = prepare_xy(test_sequences)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nExample input IDs:", X_train[0])
print("Target word ID:", y_train[0])
print("Target word:", tokenizer.index_word.get(int(y_train[0]), "<unknown>"))


X_train shape: (89620, 20)
y_train shape: (89620,)
X_test shape: (9995, 20)
y_test shape: (9995,)

Example input IDs: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0 1961]
Target word ID: 4236
Target word: gutenberg's


In [6]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.keras.utils.set_random_seed(42)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5, name="top5_accuracy"
        )
    ]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=8,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 51s 76ms/step - accuracy: 0.0541 - loss: 6.7433 - top5_accuracy: 0.1642 - val_accuracy: 0.0483 - val_loss: 6.3527 - val_top5_accuracy: 0.1627
Epoch 2/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 52s 82ms/step - accuracy: 0.0630 - loss: 6.2094 - top5_accuracy: 0.1848 - val_accuracy: 0.0622 - val_loss: 6.2264 - val_top5_accuracy: 0.1953
Epoch 3/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 77s 73ms/step - accuracy: 0.0766 - loss: 6.0167 - top5_accuracy: 0.2114 - val_accuracy: 0.0733 - val_loss: 6.1162 - val_top5_accuracy: 0.2215
Epoch 4/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 81s 73ms/step - accuracy: 0.0872 - loss: 5.8552 - top5_accuracy: 0.2362 - val_accuracy: 0.0894 - val_loss: 6.0059 - val_top5_accuracy: 0.2407
Epoch 5/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 81s 72ms/step - accuracy: 0.1050 - loss: 5.6897 - top5_accuracy: 0.2570 - val_accuracy: 0.1053 - val_loss: 5.9151 - val_top5_accuracy: 0.2661
Epoch 6/8
631/631 ━━━━━━━━━━━━━━━━━━━━ 82s 72ms/step - accuracy: 0.1160 - loss: 5.5492 - top5_a

In [8]:

test_loss, top1_accuracy, top5_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nMODEL EVALUATION RESULTS")
print("------------------------")
print(f"Test Loss: {test_loss:.4f}")
print(f"Top-1 Accuracy: {top1_accuracy * 100:.2f}%")
print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1173 - loss: 6.6477 - top5_accuracy: 0.2537

MODEL EVALUATION RESULTS
------------------------
Test Loss: 6.6477
Top-1 Accuracy: 11.73%
Top-5 Accuracy: 25.37%


In [9]:

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

phrase_input = widgets.Text(
    value="the mom is",
    placeholder="Enter a phrase...",
    description="Input:",
    layout=widgets.Layout(width="75%")
)

predict_button = widgets.Button(
    description="Find Next Word",
    button_style="primary",
    icon="search"
)

output = widgets.Output()

def predict_next_word(_):
    with output:
        clear_output(wait=True)

        phrase = phrase_input.value.strip().lower()

        if not phrase:
            print("Please enter a phrase.")
            return

        # Convert input text into word IDs
        token_ids = tokenizer.texts_to_sequences([phrase])[0]

        if not token_ids:
            print("No known words found. Try another phrase.")
            return

        # Prepare the input for the trained model
        padded_input = pad_sequences(
            [token_ids],
            maxlen=MAX_LEN,
            padding="pre",
            truncating="pre"
        )

        # Predict probability of each vocabulary word
        probabilities = model.predict(
            padded_input, verbose=0
        )[0]

        # Return the five highest-probability valid word IDs
        top_ids = np.argsort(probabilities)[::-1]
        top_ids = [
            int(i) for i in top_ids
            if 0 < i < vocab_size
        ][:5]

        print("Input phrase:", phrase)
        print("\nTop predicted next words:")

        for rank, word_id in enumerate(top_ids, start=1):
            word = tokenizer.index_word.get(word_id, "<unknown>")
            probability = probabilities[word_id] * 100
            print(f"{rank}. {word}: {probability:.2f}%")

predict_button.on_click(predict_next_word)

display(
    widgets.VBox([
        phrase_input,
        predict_button,
        output
    ])
)


Input phrase: the mom is

Top predicted next words:
1. a: 10.79%
2. the: 9.01%
3. his: 1.97%
4. in: 1.88%
5. one: 1.73%


In [10]:

import pickle

model.save("/content/next_word_lstm.keras")

with open("/content/next_word_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully!")


Model and tokenizer saved successfully!
